# Method16 Tangram score-Top100/all-cells workflow

The analysis is consolidated in one code cell. All plotting is consolidated in the final code cell. Run the cells in order.


## Analysis


In [ ]:
#!/usr/bin/env python3
"""Method16 use_X rerun: score-ranked significant positive top-100 markers and all cells.

Scientific flow is copied from
``tangram_pseudobulk_spot_scores_and_spearman_use_X.ipynb``.  Only two
reference-construction settings differ:

1. Keep ``pvals_adj < 0.05`` and ``logfoldchanges > 0``, then select the
   top 100 genes by descending ``score`` only.  Do not use p-values or
   adjusted p-values for ordering.
2. Every available cell in each subtype contributes to the pseudobulk mean;
   there is no per-subtype cell cap or random subsampling.
"""

from __future__ import annotations

import json
import math
import platform
import re
import time
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy
from scipy import sparse, stats
from statsmodels.stats.multitest import multipletests


BASE = Path("/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2")
OUTPUT_ROOT = Path.cwd()
OUTDIR = OUTPUT_ROOT
SPATIAL_DIR = (
    BASE
    / "spatial/cm_assignment_analysis/assign_nonepi_cm/mendeley_nonepi_cm_h5ad_9x9KL"
)
SC_ANNOTATED = BASE / "cm_epi_analysis/adata_anno_cell_subtype_re.h5ad"
H_DF_PATH = (
    BASE
    / "cm_epi_analysis/balanced_joint_nmf_outputs/joint_cm/tables/H_df.csv"
)
NONEPI_DEG_DIR = (
    BASE / "spatial/cm_assignment_analysis/assign_nonepi_cm/cellsubtype_degs"
)
EPI_DEG_DIR = BASE / "spatial/cm_assignment_analysis/assign_epi"
MWU_PATH = (
    BASE
    / "cm_epi_analysis/balanced_joint_nmf_outputs/joint_cm/mwu/boxplots/tumor/allCM_MWU_stats_with_median.csv"
)

SAMPLES = [
    "h46t",
    "j38t",
    "r114t",
    "r29t",
    "r51t",
    "rcor",
    "rmed",
    "s15t",
    "x49t",
    "x98t",
    "y12t",
    "y27t",
    "y7t",
    "z43t",
]
TOP_MARKERS_PER_SUBTYPE = 100
RANDOM_SEED = 42
ALPHA = 0.05
METHOD = "method16_tangram_pseudobulk_subtype_mapping_score_top100_all_cells_use_X"



def direction(value: float) -> str:
    if pd.isna(value):
        return "NA"
    return "positive" if value > 0 else ("negative" if value < 0 else "zero")


def cm_sort(values: list[str]) -> list[str]:
    def key(value: str) -> int:
        match = re.search(r"joint_(\d+)", str(value))
        return int(match.group(1)) if match else 999

    return sorted(values, key=key)


def load_marker_genes(
    path: Path | None, n: int = TOP_MARKERS_PER_SUBTYPE
) -> list[str]:
    if path is None or not path.exists():
        return []
    table = pd.read_csv(path)
    if "gene" not in table.columns:
        return []
    required = {"pvals_adj", "logfoldchanges", "score"}
    missing = required.difference(table.columns)
    if missing:
        raise KeyError(f"{path}: missing required DEG columns {sorted(missing)}")
    table["pvals_adj"] = pd.to_numeric(table["pvals_adj"], errors="raise")
    table["logfoldchanges"] = pd.to_numeric(
        table["logfoldchanges"], errors="raise"
    )
    table["score"] = pd.to_numeric(table["score"], errors="raise")
    table = table.loc[
        (table["pvals_adj"] < ALPHA) & (table["logfoldchanges"] > 0)
    ].copy()
    # Stable sorting preserves the source CSV order for exact score ties.
    table = table.sort_values("score", ascending=False, kind="mergesort")
    return table["gene"].astype(str).drop_duplicates().head(n).tolist()


def deg_path_for_subtype(subtype: str) -> Path | None:
    epithelial_path = EPI_DEG_DIR / f"{subtype}_degs_epi.csv"
    if epithelial_path.exists():
        return epithelial_path
    hits = sorted(NONEPI_DEG_DIR.glob(f"{subtype}_degs_*.csv"))
    return hits[0] if hits else None


def load_h_and_epi() -> tuple[pd.DataFrame, list[str]]:
    h_df = pd.read_csv(H_DF_PATH, index_col=0)
    h_df.index = h_df.index.astype(str)
    h_df.columns = h_df.columns.astype(str)
    epi_labels = sorted(
        path.name.replace("_degs_epi.csv", "")
        for path in EPI_DEG_DIR.glob("Epi_*_degs_epi.csv")
    )
    return h_df, epi_labels


def build_lograw_reference(
    table_dir: Path,
) -> tuple[pd.DataFrame, list[str], dict[str, list[str]], pd.DataFrame]:
    """Build subtype means from all cells using the annotated object's raw matrix."""
    h_df, epi_labels = load_h_and_epi()
    subtypes = list(dict.fromkeys(h_df.columns.tolist() + epi_labels))
    marker_genes = {
        subtype: load_marker_genes(deg_path_for_subtype(subtype))
        for subtype in subtypes
    }

    adata = sc.read_h5ad(SC_ANNOTATED, backed="r")
    try:
        if adata.raw is None:
            raise ValueError(f"{SC_ANNOTATED} has no adata.raw")
        genes = sorted(
            {
                gene
                for subtype_genes in marker_genes.values()
                for gene in subtype_genes
                if gene in adata.raw.var_names
            }
        )
        if len(genes) < 30:
            raise RuntimeError(f"Too few marker genes in scRNA raw: {len(genes)}")
        raw_gene_index = np.asarray(adata.raw.var_names.get_indexer(genes), dtype=int)
        if np.any(raw_gene_index < 0):
            raise ValueError("Marker/raw gene indexing failed")

        obs = adata.obs[["cell_subtype"]].copy()
        subtype_values = obs["cell_subtype"].astype(str)
        reference_rows: list[np.ndarray] = []
        cell_rows: list[dict[str, object]] = []
        for subtype in subtypes:
            names = obs.index[subtype_values.to_numpy() == subtype].to_numpy()
            n_total = len(names)
            if n_total == 0:
                values = np.zeros(len(genes), dtype=float)
            else:
                # Keep the original use_X implementation and remove only the
                # former random 700-cell cap: every subtype cell is included.
                matrix = adata.raw[names, :].X
                matrix = matrix[:, raw_gene_index]
                values = (
                    np.asarray(matrix.mean(axis=0)).ravel()
                    if sparse.issparse(matrix)
                    else np.asarray(matrix).mean(axis=0)
                )
            reference_rows.append(np.asarray(values, dtype=float))
            cell_rows.append(
                {
                    "subtype": subtype,
                    "n_cells_total": n_total,
                    "n_cells_used": n_total,
                    "all_cells_used": True,
                }
            )
            print(
                f"[reference] {subtype}: cells={n_total}, genes={len(genes)}",
                flush=True,
            )
    finally:
        adata.file.close()

    signature = pd.DataFrame(reference_rows, index=subtypes, columns=genes)
    signature.to_csv(table_dir / "reference_signature_lograw_mean.csv")
    pd.DataFrame(cell_rows).to_csv(
        table_dir / "reference_cells_used.csv", index=False
    )
    pd.DataFrame(
        {
            "subtype": list(marker_genes),
            "n_markers_requested": TOP_MARKERS_PER_SUBTYPE,
            "n_markers_available": [len(marker_genes[key]) for key in marker_genes],
            "marker_genes": [";".join(marker_genes[key]) for key in marker_genes],
            "source_deg_csv": [
                str(deg_path_for_subtype(key)) for key in marker_genes
            ],
        }
    ).to_csv(table_dir / "marker_genes_used.csv", index=False)
    pd.DataFrame({"gene": genes}).to_csv(
        table_dir / "tangram_reference_genes.csv", index=False
    )
    return h_df, epi_labels, marker_genes, signature


def dense(matrix):
    return matrix.toarray() if sparse.issparse(matrix) else np.asarray(matrix)


def read_spatial(
    sample: str, genes: list[str]
) -> tuple[ad.AnnData, list[str], object, pd.DataFrame]:
    adata = sc.read_h5ad(SPATIAL_DIR / f"adata_{sample}_CM.h5ad")
    present = [gene for gene in genes if gene in adata.var_names]
    matrix = adata[:, present].X  # unchanged: existing log-normalized spatial X
    obs = adata.obs[["array_row", "array_col", "sample"]].copy()
    obs.index = adata.obs_names.astype(str)
    return adata, present, matrix, obs


def aggregate_cm_epi(
    abundance: pd.DataFrame, h_df: pd.DataFrame, epi_labels: list[str]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    cm_subtypes = [column for column in h_df.columns if column in abundance.columns]
    weights = h_df[cm_subtypes].T
    weights = weights.div(weights.sum(axis=0).replace(0, np.nan), axis=1).fillna(0)
    cm_df = pd.DataFrame(
        abundance[cm_subtypes].to_numpy().dot(weights.to_numpy()),
        index=abundance.index,
        columns=h_df.index,
    )

    epi_columns = [label for label in epi_labels if label in abundance.columns]
    epi_abundance = abundance[epi_columns].astype(float).copy()
    epi_total = epi_abundance.sum(axis=1)
    epi_fraction = epi_abundance.div(
        epi_total.replace(0, np.nan), axis=0
    ).fillna(0.0)
    positive = epi_total > 0
    if positive.any() and not np.allclose(
        epi_fraction.loc[positive].sum(axis=1).to_numpy(),
        1.0,
        rtol=1e-6,
        atol=1e-8,
    ):
        raise ValueError("EPIfrac row-normalization failed")
    return cm_df, epi_fraction


def save_scores(
    sample: str,
    table_dir: Path,
    obs: pd.DataFrame,
    cm_df: pd.DataFrame,
    epi_fraction: pd.DataFrame,
) -> Path:
    spot = pd.concat(
        [
            obs,
            cm_df.add_prefix("CMact__"),
            epi_fraction.add_prefix("EPIfrac__"),
        ],
        axis=1,
    )
    path = table_dir / f"{sample}_{METHOD}_spot_scores.csv"
    temporary = path.with_suffix(path.suffix + ".tmp")
    spot.to_csv(temporary)
    temporary.replace(path)
    return path




def compare_to_mwu(results: pd.DataFrame, table_dir: Path) -> None:
    if not MWU_PATH.exists():
        (table_dir / "mwu_comparison_skipped.txt").write_text(
            f"Skipped because the original optional comparison table is absent: {MWU_PATH}\n",
            encoding="utf-8",
        )
        return
    mwu = pd.read_csv(MWU_PATH)
    mwu["mwu_direction"] = mwu["delta_high_minus_low"].map(direction)
    mwu = mwu[mwu["MWU_qvalue"] < ALPHA].copy()
    mwu["pair"] = mwu["CM"].astype(str) + "|" + mwu["epi_subtype"].astype(str)
    result = results.copy()
    result["pair"] = (
        result["CM"].astype(str) + "|" + result["epi_subtype"].astype(str)
    )
    significant = result[result["q_value_bh"] < ALPHA].copy()
    merged = mwu[
        [
            "pair",
            "CM",
            "epi_subtype",
            "MWU_qvalue",
            "delta_high_minus_low",
            "mwu_direction",
        ]
    ].merge(
        significant[["pair", "p_value", "q_value_bh", "direction"]],
        on="pair",
        how="inner",
    )
    merged = merged.rename(
        columns={
            "delta_high_minus_low": "mwu_delta_high_minus_low",
            "direction": "spatial_direction",
        }
    )
    merged["direction_consistent"] = merged["mwu_direction"].eq(
        merged["spatial_direction"]
    )
    merged["method"] = METHOD
    merged.to_csv(table_dir / f"{METHOD}_mwu_sig_intersection.csv", index=False)


def stouffer_results(score_files: list[Path], table_dir: Path):
    rows: list[dict[str, object]] = []
    for path in score_files:
        table = pd.read_csv(path, index_col=0)
        cm_columns = [column for column in table if column.startswith("CMact__")]
        epi_columns = [column for column in table if column.startswith("EPIfrac__")]
        sample = str(table["sample"].iloc[0])
        for cm_column in cm_columns:
            for epi_column in epi_columns:
                x = table[cm_column].astype(float)
                y = table[epi_column].astype(float)
                valid = x.notna() & y.notna()
                if (
                    valid.sum() < 10
                    or x[valid].nunique() < 3
                    or y[valid].nunique() < 3
                ):
                    rho, p_value = np.nan, np.nan
                else:
                    rho, p_value = stats.spearmanr(x[valid], y[valid])
                rows.append(
                    {
                        "sample": sample,
                        "CM": cm_column.replace("CMact__", ""),
                        "epi_subtype": epi_column.replace("EPIfrac__", ""),
                        "spearman_rho": rho,
                        "p_value": p_value,
                    }
                )
    per_sample = pd.DataFrame(rows)
    per_sample.to_csv(table_dir / f"{METHOD}_per_sample_spearman.csv", index=False)

    pooled_rows: list[dict[str, object]] = []
    valid_per_sample = per_sample.dropna(subset=["spearman_rho", "p_value"])
    for (cm, epi), subset in valid_per_sample.groupby(
        ["CM", "epi_subtype"], sort=False
    ):
        signed_z = []
        for row in subset.itertuples(index=False):
            p_value = max(float(row.p_value), 1e-300)
            sign = 1 if float(row.spearman_rho) >= 0 else -1
            signed_z.append(stats.norm.isf(p_value / 2) * sign)
        combined_z = np.sum(signed_z) / math.sqrt(len(signed_z))
        combined_p = 2 * stats.norm.sf(abs(combined_z))
        mean_rho = float(subset["spearman_rho"].mean())
        pooled_rows.append(
            {
                "CM": cm,
                "epi_subtype": epi,
                "mean_sample_spearman_rho": mean_rho,
                "signed_z": combined_z,
                "p_value": combined_p,
                "n_samples": len(subset),
                "direction": direction(mean_rho),
            }
        )
    results = pd.DataFrame(pooled_rows)
    if len(results):
        results["q_value_bh"] = multipletests(
            results["p_value"].fillna(1.0), method="fdr_bh"
        )[1]
        results["significant_q05"] = results["q_value_bh"] < ALPHA
    results.to_csv(table_dir / f"{METHOD}_results.csv", index=False)
    compare_to_mwu(results, table_dir)
    return results


def write_run_metadata(table_dir: Path, extra: dict[str, object] | None = None) -> None:
    metadata: dict[str, object] = {
        "template_notebook": str(
            BASE
            / "spatial/cm_assignment_analysis_method16_tangram_pseudobulk_subtype_mapping/tangram_pseudobulk_spot_scores_and_spearman_use_X.ipynb"
        ),
        "method": METHOD,
        "top_markers_per_subtype": TOP_MARKERS_PER_SUBTYPE,
        "marker_filter": "pvals_adj < 0.05 and logfoldchanges > 0",
        "marker_ordering": "score descending only; stable source order for exact ties",
        "p_value_used_for_marker_ordering": False,
        "pvals_adj_used_for_marker_filtering": True,
        "cell_sampling": "all cells per subtype; no cap; no subsampling",
        "single_cell_expression": "adata_anno_cell_subtype_re.h5ad adata.raw",
        "spatial_expression": "existing log-normalized .X",
        "tangram_mode": "cells",
        "device": "cuda:0",
        "num_epochs": 350,
        "learning_rate": 0.05,
        "random_state": RANDOM_SEED,
        "epi_fraction": "row-normalized within projected epithelial subtype abundance",
        "samples": SAMPLES,
    }
    if extra:
        metadata.update(extra)
    (table_dir / "run_parameters.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    (table_dir / "package_versions.txt").write_text(
        "\n".join(
            [
                f"python={platform.python_version()}",
                f"anndata={ad.__version__}",
                f"scanpy={sc.__version__}",
                f"numpy={np.__version__}",
                f"pandas={pd.__version__}",
                f"scipy={scipy.__version__}",
            ]
        )
        + "\n",
        encoding="utf-8",
    )


def run_tangram_analysis() -> None:
    import tangram as tg
    import torch

    table_dir = OUTDIR / "tables"
    figure_dir = OUTDIR / "figures"
    table_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for this unchanged use_X Tangram run")

    started = time.time()
    print(f"[environment] tangram={tg.__version__}, torch={torch.__version__}")
    print(f"[environment] gpu={torch.cuda.get_device_name(0)}")
    h_df, epi_labels, _, signature = build_lograw_reference(table_dir)
    genes = signature.columns.tolist()
    subtypes = signature.index.tolist()
    write_run_metadata(
        table_dir,
        {
            "n_subtypes": len(subtypes),
            "n_reference_genes": len(genes),
            "n_cm": h_df.shape[0],
            "n_epi_subtypes": len(epi_labels),
            "tangram_version": tg.__version__,
            "torch_version": torch.__version__,
        },
    )

    adata_sc = ad.AnnData(signature.to_numpy(dtype=np.float32))
    adata_sc.obs_names = subtypes
    adata_sc.var_names = genes
    adata_sc.obs["cell_subtype"] = subtypes
    score_files: list[Path] = []
    manifest_rows: list[dict[str, object]] = []

    for sample in SAMPLES:
        sample_started = time.time()
        print(f"[{METHOD}] START {sample}", flush=True)
        adata, present, spatial_expression, obs = read_spatial(sample, genes)
        try:
            adata_sp = ad.AnnData(dense(spatial_expression).astype(np.float32))
            adata_sp.obs_names = adata.obs_names.astype(str)
            adata_sp.var_names = present
            adata_sp.obs = obs.copy()
            common = [gene for gene in present if gene in adata_sc.var_names]
            asc = adata_sc[:, common].copy()
            asp = adata_sp[:, common].copy()
            tg.pp_adatas(asc, asp, genes=common)
            mapper = tg.map_cells_to_space(
                asc,
                asp,
                mode="cells",
                device="cuda:0",
                num_epochs=350,
                learning_rate=0.05,
                random_state=42,
                verbose=False,
            )
            tg.project_cell_annotations(mapper, asp, annotation="cell_subtype")
            abundance = asp.obsm["tangram_ct_pred"].copy()
            if not isinstance(abundance, pd.DataFrame):
                abundance = pd.DataFrame(
                    abundance, index=adata.obs_names.astype(str), columns=subtypes
                )
            abundance.index = adata.obs_names.astype(str)
            abundance = abundance.reindex(columns=subtypes).fillna(0)
            cm_df, epi_fraction = aggregate_cm_epi(abundance, h_df, epi_labels)
            score_path = save_scores(
                sample, table_dir, obs, cm_df, epi_fraction
            )
            score_files.append(score_path)
            manifest_rows.append(
                {
                    "sample": sample,
                    "status": "completed",
                    "n_spatial_obs": adata.n_obs,
                    "n_common_genes": len(common),
                    "score_csv": str(score_path),
                    "elapsed_seconds": round(time.time() - sample_started, 3),
                }
            )
            print(
                f"[{METHOD}] DONE {sample} genes={len(common)} "
                f"elapsed={time.time() - sample_started:.1f}s",
                flush=True,
            )
            del mapper, abundance, cm_df, epi_fraction, asc, asp, adata_sp
            torch.cuda.empty_cache()
        finally:
            del adata
        pd.DataFrame(manifest_rows).to_csv(
            table_dir / "sample_run_manifest.csv", index=False
        )

    results = stouffer_results(score_files, table_dir)
    summary = {
        "status": "completed",
        "n_completed_samples": len(score_files),
        "n_expected_samples": len(SAMPLES),
        "n_reference_subtypes": len(subtypes),
        "n_reference_genes": len(genes),
        "n_cm_epi_pairs": len(results),
        "n_significant_q05": int(results.get("significant_q05", pd.Series(dtype=bool)).sum()),
        "elapsed_seconds": round(time.time() - started, 3),
    }
    (table_dir / "run_summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)


# Percentile-quadrant Fisher analysis

import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact, norm
from statsmodels.stats.multitest import multipletests


ROOT = OUTPUT_ROOT
TABLE_DIR = ROOT / "tables"
METHOD = "method16_tangram_pseudobulk_subtype_mapping_score_top100_all_cells_use_X"
THRESHOLD = 0.5
EXCLUDE_12_SAMPLE = {"rmed", "rcor"}


def percentile_rank(values: pd.Series) -> pd.Series:
    output = pd.Series(np.nan, index=values.index, dtype=float)
    valid = pd.to_numeric(values, errors="coerce").dropna()
    if valid.empty:
        return output.fillna(0.0)
    if valid.nunique() <= 1:
        output.loc[valid.index] = 0.0
        return output.fillna(0.0)
    output.loc[valid.index] = valid.rank(method="average", pct=True)
    return output.fillna(0.0)


def quadrant_counts(cm_percentile: pd.Series, epi_percentile: pd.Series):
    cm_high = cm_percentile >= THRESHOLD
    epi_high = epi_percentile >= THRESHOLD
    return {
        "both_high": int((cm_high & epi_high).sum()),
        "cm_high_epi_low": int((cm_high & ~epi_high).sum()),
        "cm_low_epi_high": int((~cm_high & epi_high).sum()),
        "both_low": int((~cm_high & ~epi_high).sum()),
    }


def fisher_from_counts(a: int, b: int, c: int, d: int):
    odds_ratio, p_value = fisher_exact([[a, b], [c, d]], alternative="two-sided")
    return float(odds_ratio), float(p_value)


def signed_z_from_fisher(a: int, b: int, c: int, d: int, p_value: float):
    if not np.isfinite(p_value):
        return np.nan
    clipped = min(max(float(p_value), 1e-300), 1.0)
    cross_product_difference = a * d - b * c
    sign = (
        1.0
        if cross_product_difference > 0
        else (-1.0 if cross_product_difference < 0 else 0.0)
    )
    return float(sign * norm.isf(clipped / 2.0))


def add_bh_qvalue(table: pd.DataFrame):
    output = table.copy()
    valid = output["p_value"].notna()
    output["q_value_bh"] = np.nan
    if valid.any():
        output.loc[valid, "q_value_bh"] = multipletests(
            output.loc[valid, "p_value"].astype(float), method="fdr_bh"
        )[1]
    return output


def pooled_table(pooled_counts, label: str):
    rows = []
    for (cm, epi), counts in sorted(pooled_counts.items()):
        a = counts["both_high"]
        b = counts["cm_high_epi_low"]
        c = counts["cm_low_epi_high"]
        d = counts["both_low"]
        odds_ratio, p_value = fisher_from_counts(a, b, c, d)
        signed_z = signed_z_from_fisher(a, b, c, d, p_value)
        rows.append(
            {
                "analysis": label,
                "CM": cm,
                "epi_subtype": epi,
                "threshold": THRESHOLD,
                "n_both_high": a,
                "n_cm_high_epi_low": b,
                "n_cm_low_epi_high": c,
                "n_both_low": d,
                "n_spots": int(a + b + c + d),
                "odds_ratio": odds_ratio,
                "p_value": p_value,
                "signed_z": signed_z,
                "direction": (
                    "positive"
                    if signed_z > 0
                    else ("negative" if signed_z < 0 else "neutral")
                ),
            }
        )
    return add_bh_qvalue(pd.DataFrame(rows))


def stouffer_table(per_sample: pd.DataFrame, samples: set[str], label: str):
    subset = per_sample.loc[per_sample["sample"].isin(samples)].copy()
    rows = []
    for (cm, epi), group in subset.groupby(["CM", "epi_subtype"], sort=True):
        z_values = (
            group["signed_z"]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .astype(float)
        )
        if z_values.empty:
            combined_z = np.nan
            p_value = np.nan
        else:
            combined_z = float(z_values.sum() / np.sqrt(len(z_values)))
            p_value = float(2.0 * norm.sf(abs(combined_z)))
        rows.append(
            {
                "analysis": label,
                "CM": cm,
                "epi_subtype": epi,
                "threshold": THRESHOLD,
                "n_samples": int(len(z_values)),
                "combined_signed_z": combined_z,
                "p_value": p_value,
                "direction": (
                    "positive"
                    if combined_z > 0
                    else ("negative" if combined_z < 0 else "neutral")
                ),
            }
        )
    return add_bh_qvalue(pd.DataFrame(rows))


def run_percentile_quadrant_fisher():
    score_files = sorted(TABLE_DIR.glob(f"*_{METHOD}_spot_scores.csv"))
    if len(score_files) != 14:
        raise ValueError(f"Expected 14 score files, found {len(score_files)}")

    per_sample_rows = []
    pooled_counts_14 = {}
    pooled_counts_12 = {}
    for score_path in score_files:
        score = pd.read_csv(score_path, index_col=0)
        samples = score["sample"].dropna().astype(str).unique().tolist()
        if len(samples) != 1:
            raise ValueError(f"{score_path}: expected one sample, found {samples}")
        sample = samples[0]
        cm_names = [
            column.split("__", 1)[1]
            for column in score.columns
            if column.startswith("CMact__")
        ]
        epi_names = [
            column.split("__", 1)[1]
            for column in score.columns
            if column.startswith("EPIfrac__")
        ]
        cm_percentiles = {
            cm: percentile_rank(score[f"CMact__{cm}"]) for cm in cm_names
        }
        epi_percentiles = {
            epi: percentile_rank(score[f"EPIfrac__{epi}"]) for epi in epi_names
        }

        for cm in cm_names:
            for epi in epi_names:
                counts = quadrant_counts(cm_percentiles[cm], epi_percentiles[epi])
                a = counts["both_high"]
                b = counts["cm_high_epi_low"]
                c = counts["cm_low_epi_high"]
                d = counts["both_low"]
                odds_ratio, p_value = fisher_from_counts(a, b, c, d)
                signed_z = signed_z_from_fisher(a, b, c, d, p_value)
                per_sample_rows.append(
                    {
                        "sample": sample,
                        "CM": cm,
                        "epi_subtype": epi,
                        "threshold": THRESHOLD,
                        "n_both_high": a,
                        "n_cm_high_epi_low": b,
                        "n_cm_low_epi_high": c,
                        "n_both_low": d,
                        "n_spots": int(a + b + c + d),
                        "odds_ratio": odds_ratio,
                        "p_value": p_value,
                        "signed_z": signed_z,
                        "direction": (
                            "positive"
                            if signed_z > 0
                            else ("negative" if signed_z < 0 else "neutral")
                        ),
                    }
                )
                key = (cm, epi)
                destinations = (
                    [pooled_counts_14]
                    if sample in EXCLUDE_12_SAMPLE
                    else [pooled_counts_14, pooled_counts_12]
                )
                for destination in destinations:
                    if key not in destination:
                        destination[key] = {
                            "both_high": 0,
                            "cm_high_epi_low": 0,
                            "cm_low_epi_high": 0,
                            "both_low": 0,
                        }
                    for count_key, value in counts.items():
                        destination[key][count_key] += int(value)

    per_sample = add_bh_qvalue(pd.DataFrame(per_sample_rows))
    pooled14 = pooled_table(pooled_counts_14, "pooled_spot_14samples")
    pooled12 = pooled_table(
        pooled_counts_12, "pooled_spot_12samples_no_rmed_rcor"
    )
    all_samples = set(per_sample["sample"].astype(str))
    samples12 = all_samples.difference(EXCLUDE_12_SAMPLE)
    stouffer14 = stouffer_table(
        per_sample, all_samples, "sample_stouffer_14samples"
    )
    stouffer12 = stouffer_table(
        per_sample, samples12, "sample_stouffer_12samples_no_rmed_rcor"
    )

    outputs = {
        "per_sample": per_sample,
        "pooled_spot_14samples": pooled14,
        "pooled_spot_12samples_no_rmed_rcor": pooled12,
        "sample_stouffer_14samples": stouffer14,
        "sample_stouffer_12samples_no_rmed_rcor": stouffer12,
    }
    for label, table in outputs.items():
        table.to_csv(
            TABLE_DIR / f"{METHOD}_percentile_quadrant_fisher_{label}.csv",
            index=False,
        )

    summary = {
        "status": "completed",
        "plotting_performed": False,
        "threshold": THRESHOLD,
        "n_per_sample_tests": len(per_sample),
        "n_pairs": len(stouffer14),
        "per_sample_significant_q05": int((per_sample["q_value_bh"] < 0.05).sum()),
        "pooled14_significant_q05": int((pooled14["q_value_bh"] < 0.05).sum()),
        "pooled12_significant_q05": int((pooled12["q_value_bh"] < 0.05).sum()),
        "stouffer14_significant_q05": int((stouffer14["q_value_bh"] < 0.05).sum()),
        "stouffer12_significant_q05": int((stouffer12["q_value_bh"] < 0.05).sum()),
    }
    (TABLE_DIR / f"{METHOD}_percentile_quadrant_fisher_summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print(json.dumps(summary, ensure_ascii=False, indent=2))


run_tangram_analysis()
run_percentile_quadrant_fisher()


## Plotting


In [ ]:
def plot_heatmap(results: pd.DataFrame, figure_dir: Path) -> None:
    if results.empty:
        return
    matrix = results.pivot(
        index="epi_subtype", columns="CM", values="mean_sample_spearman_rho"
    )
    matrix = matrix.reindex(
        index=sorted(matrix.index), columns=cm_sort(list(matrix.columns))
    )
    values = matrix.to_numpy(dtype=float)
    vmax = np.nanmax(np.abs(values)) if np.isfinite(values).any() else 1.0
    fig, ax = plt.subplots(
        figsize=(
            max(7, 0.45 * matrix.shape[1] + 2),
            max(4, 0.35 * matrix.shape[0] + 1.8),
        )
    )
    sns.heatmap(
        matrix,
        cmap="coolwarm",
        center=0,
        vmin=-vmax,
        vmax=vmax,
        linewidths=0.2,
        cbar_kws={"label": "mean sample Spearman rho"},
        ax=ax,
    )
    ax.set_title(METHOD)
    fig.tight_layout()
    fig.savefig(figure_dir / f"{METHOD}_heatmap.pdf", bbox_inches="tight")
    fig.savefig(figure_dir / f"{METHOD}_heatmap.svg", bbox_inches="tight")
    plt.close(fig)


# Fisher signed-Stouffer heatmaps

import re
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


METHOD_DIR = OUTPUT_ROOT
TABLE_DIR = METHOD_DIR / "tables"
FIGURE_DIR = METHOD_DIR / "figures" / "percentile_quadrant_fisher_heatmaps"
METHOD = "method16_tangram_pseudobulk_subtype_mapping_score_top100_all_cells_use_X"

STOUFFER_TABLES = [
    (
        TABLE_DIR
        / f"{METHOD}_percentile_quadrant_fisher_sample_stouffer_14samples.csv",
        f"{METHOD}_percentile_quadrant_fisher_sample_stouffer_14samples_signedZ_qstars",
        "Percentile-quadrant Fisher Stouffer (14 samples)",
    ),
    (
        TABLE_DIR
        / f"{METHOD}_percentile_quadrant_fisher_sample_stouffer_12samples_no_rmed_rcor.csv",
        f"{METHOD}_percentile_quadrant_fisher_sample_stouffer_12samples_no_rmed_rcor_signedZ_qstars",
        "Percentile-quadrant Fisher Stouffer (12 tumor samples)",
    ),
]

plt.rcParams.update(
    {
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "text.usetex": False,
        "font.family": "DejaVu Sans",
        "font.size": 8,
        "axes.titlesize": 9,
    }
)


def cm_sort_key(value: str) -> tuple[int, str]:
    match = re.search(r"joint_(\d+)", str(value))
    return (int(match.group(1)) if match else 999, str(value))


def q_label(q_value: float) -> str:
    if not np.isfinite(q_value):
        return "ns"
    if q_value < 0.001:
        return "***"
    if q_value < 0.01:
        return "**"
    if q_value < 0.05:
        return "*"
    return "ns"


def ordered_matrices(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    z_matrix = df.pivot(
        index="epi_subtype", columns="CM", values="combined_signed_z"
    )
    q_matrix = df.pivot(index="epi_subtype", columns="CM", values="q_value_bh")
    row_order = sorted(z_matrix.index)
    column_order = sorted(z_matrix.columns, key=cm_sort_key)
    return (
        z_matrix.reindex(index=row_order, columns=column_order),
        q_matrix.reindex(index=row_order, columns=column_order),
    )


def q_annotation(z_matrix: pd.DataFrame, q_matrix: pd.DataFrame) -> pd.DataFrame:
    labels = pd.DataFrame("", index=z_matrix.index, columns=z_matrix.columns)
    for epi in labels.index:
        for cm in labels.columns:
            labels.loc[epi, cm] = q_label(float(q_matrix.loc[epi, cm]))
    return labels


def symmetric_limit(matrix: pd.DataFrame) -> float:
    values = matrix.to_numpy(dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return 1.0
    limit = float(np.max(np.abs(finite)))
    return limit if limit > 0 else 1.0


def plot_one(input_csv: Path, output_stem: str, title: str) -> dict[str, object]:
    table = pd.read_csv(input_csv)
    required = {"CM", "epi_subtype", "combined_signed_z", "q_value_bh"}
    missing = required.difference(table.columns)
    if missing:
        raise KeyError(f"{input_csv.name}: missing {sorted(missing)}")

    z_matrix, q_matrix = ordered_matrices(table)
    annotation = q_annotation(z_matrix, q_matrix)
    limit = symmetric_limit(z_matrix)
    fig_width = max(7.8, 0.52 * z_matrix.shape[1] + 2.6)
    fig_height = max(4.8, 0.42 * z_matrix.shape[0] + 1.9)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    sns.heatmap(
        z_matrix,
        cmap="coolwarm",
        center=0,
        vmin=-limit,
        vmax=limit,
        linewidths=0.25,
        linecolor="white",
        annot=annotation,
        fmt="",
        annot_kws={"fontsize": 6.5, "color": "black", "linespacing": 0.9},
        cbar_kws={"label": "combined signed Z"},
        square=True,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("CM")
    ax.set_ylabel("Epithelial subtype")
    ax.tick_params(axis="x", labelrotation=90)
    ax.tick_params(axis="y", labelrotation=0)
    fig.tight_layout()

    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    pdf_path = FIGURE_DIR / f"{output_stem}.pdf"
    svg_path = FIGURE_DIR / f"{output_stem}.svg"
    fig.savefig(pdf_path, bbox_inches="tight", dpi=300)
    fig.savefig(svg_path, bbox_inches="tight", dpi=300)
    plt.close(fig)

    return {
        "source_csv": str(input_csv.relative_to(METHOD_DIR)),
        "pdf": str(pdf_path.relative_to(METHOD_DIR)),
        "svg": str(svg_path.relative_to(METHOD_DIR)),
        "n_epi_subtypes": int(z_matrix.shape[0]),
        "n_cm": int(z_matrix.shape[1]),
        "n_pairs": int(z_matrix.size),
        "n_q_lt_0p05": int((q_matrix < 0.05).sum().sum()),
    }


def plot_fisher_stouffer_heatmaps() -> None:
    rows = []
    for input_csv, output_stem, title in STOUFFER_TABLES:
        if not input_csv.exists():
            raise FileNotFoundError(input_csv)
        result = plot_one(input_csv, output_stem, title)
        rows.append(result)
        print(result["pdf"], flush=True)
        print(result["svg"], flush=True)

    manifest = pd.DataFrame(rows)
    manifest_path = TABLE_DIR / "plot_fisher_stouffer_heatmaps_manifest.csv"
    manifest.to_csv(manifest_path, index=False)
    print(manifest.to_string(index=False), flush=True)
    print(manifest_path.relative_to(METHOD_DIR), flush=True)


primary_results = pd.read_csv(TABLE_DIR / f"{METHOD}_results.csv")
plot_heatmap(primary_results, OUTPUT_ROOT / "figures")
plot_fisher_stouffer_heatmaps()



# All spatial pair plots and core-pair forest plot

import math
import re
from multiprocessing import Pool
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from statsmodels.stats.multitest import multipletests


SEED = 42
np.random.seed(SEED)

METHOD_DIR = OUTPUT_ROOT
TABLE_DIR = METHOD_DIR / "tables"
FIGURE_DIR = METHOD_DIR / "figures"
METHOD = "method16_tangram_pseudobulk_subtype_mapping_score_top100_all_cells_use_X"
SCORE_GLOB = f"*_{METHOD}_spot_scores.csv"
PER_SAMPLE_SPEARMAN = TABLE_DIR / f"{METHOD}_per_sample_spearman.csv"

RAW_DIR = FIGURE_DIR / "raw_cmact_epifrac_by_sample"
PERCENTILE_DIR = FIGURE_DIR / "percentile_quadrant_by_sample"
FOREST_DIR = FIGURE_DIR / "core_pair_per_sample_forest"
MANIFEST_PATH = TABLE_DIR / "all_sample_cm_epi_pair_plot_manifest.csv"
FOREST_TABLE_PATH = TABLE_DIR / "core_pair_per_sample_spearman_for_forest.csv"

N_WORKERS = 4
PERCENTILE_CUTOFF = 0.5

# FIXED publication core subset requested for the forest.  This does not limit
# the all-pair raw/percentile/quadrant spatial maps below.
CORE_PAIRS = [
    ("joint_01_sharedCM", "Epi_VIM"),
    ("joint_01_sharedCM", "Epi_JUN"),
    ("joint_03_sharedCM", "Epi_VIM"),
    ("joint_03_sharedCM", "Epi_JUN"),
]
FOREST_EXCLUDED_SAMPLES = {"rcor", "rmed"}

CATEGORY_ORDER = ["both_low", "epi_high_only", "cm_high_only", "both_high"]
CATEGORY_LABELS = {
    "both_low": "CM<0.5, Epi<0.5",
    "epi_high_only": "Epi>=0.5 only",
    "cm_high_only": "CM>=0.5 only",
    "both_high": "Both>=0.5",
}
CATEGORY_COLORS = {
    "both_low": "#d9d9d9",
    "epi_high_only": "#f59e0b",
    "cm_high_only": "#2b6cb0",
    "both_high": "#b91c1c",
}
CM_PERCENTILE_CMAP = LinearSegmentedColormap.from_list(
    "cm_percentile_light_to_blue",
    ["#f2f2f2", "#b7d7ea", "#4c78a8", CATEGORY_COLORS["cm_high_only"]],
)
EPI_PERCENTILE_CMAP = LinearSegmentedColormap.from_list(
    "epi_percentile_light_to_orange",
    ["#f2f2f2", "#fde6b3", "#fbbf24", CATEGORY_COLORS["epi_high_only"]],
)

plt.rcParams.update(
    {
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "font.family": "DejaVu Sans",
        "font.size": 7,
        "axes.titlesize": 8,
        "axes.labelsize": 7,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
    }
)


def safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def infer_sample(score: pd.DataFrame, score_path: Path) -> str:
    values = sorted(score["sample"].dropna().astype(str).unique().tolist())
    if len(values) == 1:
        return values[0]
    return score_path.name.split(f"_{METHOD}_", 1)[0]


def robust_limits(values: pd.Series, q=(0.01, 0.99)) -> tuple[float, float]:
    arr = (
        pd.to_numeric(values, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .to_numpy()
    )
    if arr.size == 0:
        return 0.0, 1.0
    lo, hi = np.nanquantile(arr, q)
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        lo, hi = float(np.nanmin(arr)), float(np.nanmax(arr))
    if lo == hi:
        hi = lo + 1e-9
    return float(lo), float(hi)


def plot_percentile_rank(values: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(values, errors="coerce")
    return numeric.rank(method="average", pct=True).fillna(0.0)


def classify_percentiles(cm_pct: pd.Series, epi_pct: pd.Series) -> pd.Series:
    cm_high = cm_pct >= PERCENTILE_CUTOFF
    epi_high = epi_pct >= PERCENTILE_CUTOFF
    category = pd.Series("both_low", index=cm_pct.index, dtype="object")
    category.loc[epi_high & ~cm_high] = "epi_high_only"
    category.loc[cm_high & ~epi_high] = "cm_high_only"
    category.loc[cm_high & epi_high] = "both_high"
    return category


def marker_size(x: np.ndarray, y: np.ndarray, panel_inches: float = 4.5) -> float:
    points = np.column_stack([x, y])
    if len(points) < 2:
        base = 16.0
    else:
        distances = NearestNeighbors(n_neighbors=2).fit(points).kneighbors(
            points, return_distance=True
        )[0][:, 1]
        spacing = float(np.nanmedian(distances))
        x_span = max(float(np.nanmax(x) - np.nanmin(x)), 1e-9)
        y_span = max(float(np.nanmax(y) - np.nanmin(y)), 1e-9)
        points_per_coord = min(
            panel_inches * 72.0 / x_span, panel_inches * 72.0 / y_span
        )
        base = (spacing * points_per_coord * 0.62) ** 2
    return float(np.clip(base, 3.0, 42.0))


def set_spatial_axes(ax, x: np.ndarray, y: np.ndarray) -> None:
    ax.set_box_aspect(1)
    ax.set_xlabel("array_col")
    ax.set_ylabel("array_row")
    ax.set_xlim(np.nanmin(x) - 2, np.nanmax(x) + 2)
    ax.set_ylim(np.nanmax(y) + 2, np.nanmin(y) - 2)
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)


def save_pdf_svg(fig, stem: Path) -> tuple[Path, Path]:
    stem.parent.mkdir(parents=True, exist_ok=True)
    pdf = stem.with_suffix(".pdf")
    svg = stem.with_suffix(".svg")
    fig.savefig(pdf, bbox_inches="tight", dpi=300)
    fig.savefig(svg, bbox_inches="tight", dpi=300)
    plt.close(fig)
    return pdf, svg


def plot_raw_pair(
    score: pd.DataFrame,
    sample: str,
    cm: str,
    epi: str,
    out_dir: Path,
    size: float,
) -> tuple[Path, Path]:
    x = score["array_col"].to_numpy(dtype=float)
    y = score["array_row"].to_numpy(dtype=float)
    cm_raw = pd.to_numeric(score[f"CMact__{cm}"], errors="coerce")
    epi_raw = pd.to_numeric(score[f"EPIfrac__{epi}"], errors="coerce")

    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.6), constrained_layout=True)
    cm_scatter = axes[0].scatter(
        x,
        y,
        c=cm_raw,
        s=size,
        cmap="magma",
        norm=Normalize(*robust_limits(cm_raw)),
        marker="o",
        linewidths=0,
        rasterized=True,
    )
    set_spatial_axes(axes[0], x, y)
    axes[0].set_title(f"CMact: {cm}")
    fig.colorbar(cm_scatter, ax=axes[0], fraction=0.046, pad=0.02).set_label(
        "CMact"
    )

    epi_scatter = axes[1].scatter(
        x,
        y,
        c=epi_raw,
        s=size,
        cmap="viridis",
        norm=Normalize(*robust_limits(epi_raw)),
        marker="o",
        linewidths=0,
        rasterized=True,
    )
    set_spatial_axes(axes[1], x, y)
    axes[1].set_title(f"EPIfrac: {epi}")
    fig.colorbar(epi_scatter, ax=axes[1], fraction=0.046, pad=0.02).set_label(
        "EPIfrac"
    )

    fig.suptitle(f"{sample}: {cm} | {epi}", y=1.04, fontsize=9, fontweight="bold")
    stem = out_dir / f"{sample}__{safe_name(cm)}__{safe_name(epi)}__raw_cmact_epifrac"
    return save_pdf_svg(fig, stem)


def plot_percentile_pair(
    score: pd.DataFrame,
    sample: str,
    cm: str,
    epi: str,
    out_dir: Path,
    size: float,
) -> tuple[Path, Path]:
    x = score["array_col"].to_numpy(dtype=float)
    y = score["array_row"].to_numpy(dtype=float)
    cm_pct = plot_percentile_rank(score[f"CMact__{cm}"])
    epi_pct = plot_percentile_rank(score[f"EPIfrac__{epi}"])
    category = classify_percentiles(cm_pct, epi_pct)

    fig, axes = plt.subplots(1, 3, figsize=(14.8, 4.8), constrained_layout=True)
    cm_scatter = axes[0].scatter(
        x,
        y,
        c=cm_pct,
        cmap=CM_PERCENTILE_CMAP,
        vmin=0.0,
        vmax=1.0,
        s=size,
        marker="o",
        linewidths=0,
        rasterized=True,
    )
    set_spatial_axes(axes[0], x, y)
    axes[0].set_title(f"CM percentile: {cm}")
    fig.colorbar(cm_scatter, ax=axes[0], fraction=0.046, pad=0.02).set_label(
        "CMact percentile"
    )

    epi_scatter = axes[1].scatter(
        x,
        y,
        c=epi_pct,
        cmap=EPI_PERCENTILE_CMAP,
        vmin=0.0,
        vmax=1.0,
        s=size,
        marker="o",
        linewidths=0,
        rasterized=True,
    )
    set_spatial_axes(axes[1], x, y)
    axes[1].set_title(f"Epi percentile: {epi}")
    fig.colorbar(epi_scatter, ax=axes[1], fraction=0.046, pad=0.02).set_label(
        "EPIfrac percentile"
    )

    for category_name in CATEGORY_ORDER:
        mask = category.eq(category_name).to_numpy()
        if mask.any():
            axes[2].scatter(
                x[mask],
                y[mask],
                c=CATEGORY_COLORS[category_name],
                s=size,
                marker="o",
                linewidths=0,
                rasterized=True,
            )
    set_spatial_axes(axes[2], x, y)
    axes[2].set_title("Percentile quadrant")
    handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            label=CATEGORY_LABELS[category_name],
            markerfacecolor=CATEGORY_COLORS[category_name],
            markersize=6,
        )
        for category_name in CATEGORY_ORDER
    ]
    axes[2].legend(
        handles=handles,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        frameon=True,
        fontsize=6,
        borderpad=0.35,
        borderaxespad=0.0,
    )
    fig.suptitle(f"{sample}: {cm} | {epi}", y=1.04, fontsize=9, fontweight="bold")
    stem = out_dir / (
        f"{sample}__{safe_name(cm)}__{safe_name(epi)}__percentile_quadrant_0p5"
    )
    return save_pdf_svg(fig, stem)


def process_sample(score_path: Path) -> list[dict[str, object]]:
    score = pd.read_csv(score_path, index_col=0)
    sample = infer_sample(score, score_path)
    required = {"sample", "array_row", "array_col"}
    missing = required.difference(score.columns)
    if missing:
        raise KeyError(f"{sample}: score CSV missing {sorted(missing)}")

    cm_names = sorted(
        {column.split("__", 1)[1] for column in score if column.startswith("CMact__")}
    )
    epi_names = sorted(
        {column.split("__", 1)[1] for column in score if column.startswith("EPIfrac__")}
    )
    if not cm_names or not epi_names:
        raise ValueError(f"{sample}: no CMact__/EPIfrac__ columns")

    x = score["array_col"].to_numpy(dtype=float)
    y = score["array_row"].to_numpy(dtype=float)
    size = marker_size(x, y)
    raw_sample_dir = RAW_DIR / sample
    pct_sample_dir = PERCENTILE_DIR / sample
    rows: list[dict[str, object]] = []
    print(
        f"[spatial-plots] {sample}: {len(cm_names)} CM x {len(epi_names)} Epi",
        flush=True,
    )
    for cm in cm_names:
        for epi in epi_names:
            raw_pdf, raw_svg = plot_raw_pair(
                score, sample, cm, epi, raw_sample_dir, size
            )
            pct_pdf, pct_svg = plot_percentile_pair(
                score, sample, cm, epi, pct_sample_dir, size
            )
            rows.append(
                {
                    "sample": sample,
                    "CM": cm,
                    "epi_subtype": epi,
                    "n_spots": int(len(score)),
                    "raw_pdf": str(raw_pdf.relative_to(METHOD_DIR)),
                    "raw_svg": str(raw_svg.relative_to(METHOD_DIR)),
                    "percentile_quadrant_pdf": str(pct_pdf.relative_to(METHOD_DIR)),
                    "percentile_quadrant_svg": str(pct_svg.relative_to(METHOD_DIR)),
                    "status": "ok",
                }
            )
    print(f"[spatial-plots] {sample}: done {len(rows)} pairs", flush=True)
    return rows


def plot_core_forest(score_paths: list[Path]) -> tuple[Path, Path]:
    per_sample = pd.read_csv(PER_SAMPLE_SPEARMAN)
    selected_parts = []
    for order, (cm, epi) in enumerate(CORE_PAIRS):
        hit = per_sample.loc[
            per_sample["CM"].eq(cm) & per_sample["epi_subtype"].eq(epi)
        ].copy()
        if hit.empty:
            raise KeyError(f"Missing forest pair: {cm} | {epi}")
        hit["pair_order"] = order
        hit["pair"] = f"{cm} - {epi}"
        selected_parts.append(hit)
    selected = pd.concat(selected_parts, ignore_index=True)
    selected = selected.loc[
        ~selected["sample"].astype(str).isin(FOREST_EXCLUDED_SAMPLES)
    ].copy()

    n_spots = {}
    for score_path in score_paths:
        score = pd.read_csv(score_path, index_col=0, usecols=None)
        n_spots[infer_sample(score, score_path)] = len(score)
    selected["n_spots"] = selected["sample"].astype(str).map(n_spots)

    # FIXED: for each CM-Epi pair, correct its 12 tumor-sample Spearman p-values
    # as one BH family.  Different CM-Epi pairs are corrected independently.
    selected["q_value"] = np.nan
    for _, group_index in selected.groupby(["CM", "epi_subtype"]).groups.items():
        group_index = pd.Index(group_index)
        group_p = pd.to_numeric(selected.loc[group_index, "p_value"], errors="coerce")
        valid_index = group_p.index[group_p.notna()]
        if len(valid_index):
            selected.loc[valid_index, "q_value"] = multipletests(
                group_p.loc[valid_index].astype(float), method="fdr_bh"
            )[1]
    selected["bh_family"] = "within_pair_across_12_tumor_samples"
    selected["bh_family_n"] = selected.groupby(["CM", "epi_subtype"])[
        "p_value"
    ].transform(lambda values: int(pd.to_numeric(values, errors="coerce").notna().sum()))
    selected.to_csv(FOREST_TABLE_PATH, index=False)

    rho_values = pd.to_numeric(selected["spearman_rho"], errors="coerce").dropna()
    max_abs = max(abs(float(rho_values.min())), abs(float(rho_values.max())), 0.05)
    xlim = (-1.12 * max_abs, 1.12 * max_abs)

    q_values = pd.to_numeric(selected["q_value"], errors="coerce")
    neglogq = -np.log10(q_values.clip(lower=np.nextafter(0, 1)))
    significant_values = neglogq[q_values < 0.05]
    color_min = -np.log10(0.05)
    color_max = (
        float(np.nanpercentile(significant_values, 95))
        if np.isfinite(significant_values).any()
        else color_min + 1.0
    )
    color_max = max(color_max, color_min + 0.1)
    norm = Normalize(vmin=color_min, vmax=color_max)
    cmap = plt.cm.viridis

    # FIXED requested publication layout: four core pairs in one horizontal row.
    fig, axes = plt.subplots(1, 4, figsize=(18.0, 5.8), sharex=True)
    axes = np.asarray(axes).ravel()
    for ax, (cm, epi) in zip(axes, CORE_PAIRS):
        pair = f"{cm} - {epi}"
        subset = selected.loc[selected["pair"].eq(pair)].sort_values(
            "spearman_rho"
        )
        sizes = pd.to_numeric(subset["n_spots"], errors="coerce").fillna(100.0)
        sizes = 20.0 + 80.0 * sizes / sizes.max()
        subset_q = pd.to_numeric(subset["q_value"], errors="coerce")
        subset_color = -np.log10(subset_q.clip(lower=np.nextafter(0, 1)))
        significant = subset_q < 0.05
        y_positions = np.arange(len(subset))

        ax.scatter(
            subset.loc[~significant, "spearman_rho"],
            y_positions[~significant.to_numpy()],
            s=sizes.loc[~significant],
            color="#b8b8b8",
            alpha=0.7,
            edgecolors="none",
        )
        ax.scatter(
            subset.loc[significant, "spearman_rho"],
            y_positions[significant.to_numpy()],
            s=sizes.loc[significant],
            c=subset_color.loc[significant],
            cmap=cmap,
            norm=norm,
            alpha=0.85,
            edgecolors="none",
        )
        ax.axvline(0, color="black", lw=0.8)
        ax.set_xlim(*xlim)
        ax.set_yticks(y_positions)
        ax.set_yticklabels(subset["sample"], fontsize=6)
        ax.set_title(pair, fontsize=9)
        ax.grid(axis="x", color="#dddddd", lw=0.5)
        ax.set_xlabel("Spearman rho")
        # FIXED: each panel is slightly wider than tall (width/height ~= 1.18).
        ax.set_box_aspect(0.85)

    scalar_mappable = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    scalar_mappable.set_array([])
    fig.subplots_adjust(
        left=0.055, right=0.90, top=0.87, bottom=0.14, wspace=0.42
    )
    color_ax = fig.add_axes([0.925, 0.25, 0.012, 0.52])
    colorbar = fig.colorbar(scalar_mappable, cax=color_ax)
    colorbar.set_label("-log10(BH q within pair), q < 0.05", fontsize=9)
    fig.legend(
        handles=[
            Line2D(
                [0],
                [0],
                marker="o",
                color="none",
                markerfacecolor="#b8b8b8",
                markersize=6,
                label="q >= 0.05",
            )
        ],
        loc="lower right",
        bbox_to_anchor=(0.975, 0.11),
        frameon=False,
        fontsize=8,
    )
    fig.suptitle(
        "Per-sample spatial Spearman correlations for core CM-Epi pairs",
        fontsize=12,
        fontweight="bold",
    )
    return save_pdf_svg(
        fig, FOREST_DIR / "01_spatial_core_pair_per_sample_forest_colored_by_log10q"
    )


def plot_all_pairs_and_core_forest() -> None:
    score_paths = sorted(TABLE_DIR.glob(SCORE_GLOB))
    if not score_paths:
        raise FileNotFoundError(f"No score CSVs found: {TABLE_DIR / SCORE_GLOB}")
    if not PER_SAMPLE_SPEARMAN.exists():
        raise FileNotFoundError(PER_SAMPLE_SPEARMAN)

    forest_pdf, forest_svg = plot_core_forest(score_paths)
    print(f"[forest] wrote {forest_pdf}", flush=True)
    print(f"[forest] wrote {forest_svg}", flush=True)

    rows: list[dict[str, object]] = []
    worker_count = min(N_WORKERS, len(score_paths))
    print(f"[spatial-plots] sample workers={worker_count}", flush=True)
    if worker_count == 1:
        for score_path in score_paths:
            rows.extend(process_sample(score_path))
    else:
        with Pool(processes=worker_count) as pool:
            for sample_rows in pool.imap_unordered(process_sample, score_paths):
                rows.extend(sample_rows)

    manifest = pd.DataFrame(rows).sort_values(["sample", "CM", "epi_subtype"])
    manifest.to_csv(MANIFEST_PATH, index=False)
    expected_pairs = 12 * 9
    expected_rows = len(score_paths) * expected_pairs
    if len(manifest) != expected_rows or not manifest["status"].eq("ok").all():
        raise RuntimeError(
            f"Incomplete plot manifest: rows={len(manifest)}, expected={expected_rows}"
        )
    print(f"[spatial-plots] completed rows={len(manifest)}", flush=True)
    print(f"[spatial-plots] manifest={MANIFEST_PATH}", flush=True)


plot_all_pairs_and_core_forest()
